# Armenian poetry GPT

Training notebook (Colab). Data files and the tokenizer are expected in `MyDrive`, see README.

## Settings
Hyperparameters for the current stage. For each stage I change TEXT_FILE, lr, dropout and the step counts.


In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F
import json
import os
from google.colab import drive
from tokenizers import Tokenizer


drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
TEXT_FILE = 'sevak.txt' 

tok = Tokenizer.from_file(f"{DRIVE}/tokenizer_8192.json")
torch.manual_seed(1337)
n_embed = 512
n_layer = 8
dropout = 0.2
batch_size = 64
block_size = 256
eval_iters = 20
max_iters = 400
n_head = 8
eval_interval = 20
lr = 1e-4
min_lr = 1e-5
PATIENCE = 4  #թե մոդելը քանի անգամ կարող է նախորդից ավելի բարձր val los տալ
device  = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

Mounted at /content/drive
cuda


## Tokenizer and data
BPE tokenizer (8192 tokens) and the encoded data, stored as int16 tensors on Drive.


In [2]:
stoi = tok.get_vocab()
itos = {i: s for s, i in stoi.items()}
vocab_size = tok.get_vocab_size()

encode = lambda s: tok.encode(s).ids
decode = lambda k: ''.join(itos[i] for i in k)

print(vocab_size)
print(tok.encode("Ես քեզ սիրում եմ").tokens)


8192
['Ես', ' քեզ', ' սիրում', ' եմ']


In [ ]:
# Առաջին անգամ՝ encode ու պահում Drive-ում, հաջորդ անգամներին՝ միանգամից բեռնում
DATA_PATH = f"{DRIVE}/{Path(TEXT_FILE).stem}_bpe8192.pt"

if os.path.exists(DATA_PATH):
    data = torch.load(DATA_PATH)
    print("Բեռնեցի պատրաստ data-ն")
else:
    text = Path(f"{DRIVE}/{TEXT_FILE}").read_text(encoding="utf-8")
    data = torch.tensor(encode(text), dtype=torch.int16)
    torch.save(data, DATA_PATH)
    del text
    print("Encode արեցի ու պահեցի")

print(len(data), "token")
print(decode(data[:300].tolist()))

In [ ]:
n = int(0.9 * len(data))

train_data = data[:n].to(device)
val_data = data[n:].to(device)

In [ ]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,), device=device)
    offsets = torch.arange(block_size + 1, device=device)
    chunk = data[ix[:, None] + offsets].long()   # (B, block_size+1)
    return chunk[:, :-1], chunk[:, 1:]

## Model
GPT-style decoder: 8 layers, 512 dim, 8 heads, flash attention through scaled_dot_product_attention.


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.n_head = num_heads
        self.key = nn.Linear(n_embed, n_embed, bias=False)
        self.query = nn.Linear(n_embed, n_embed, bias=False)
        self.value = nn.Linear(n_embed, n_embed, bias=False)
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        k = self.key(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=dropout if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        return self.dropout(self.proj(out))


In [7]:
class FeedForward(nn.Module):

    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
class Block(nn.Module):

    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None): # idx, target  = (B, T)
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B,T,C) + (T,C) with broadcasting
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x) #(B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, vocab_size)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



m = GPT()
m = m.to(device)


## Training
AdamW, linear lr decay, early stopping on val loss, bf16 autocast. Checkpoints go to Drive so a run can continue after a disconnect.


In [10]:
optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

In [11]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [12]:
def get_lr(it):
    new_lr = lr + (it / max_iters) * (min_lr - lr)

    return new_lr

In [ ]:
RUN = "sevak_bpe200j"
INIT_FROM = f"{DRIVE}/sevak_bpe200h_best.pth"


SAVE_PATH = f"{DRIVE}/{RUN}_best.pth"
LAST_PATH = f"{DRIVE}/{RUN}_last.pth"

if os.path.exists(LAST_PATH):
    chkpt = torch.load(LAST_PATH, map_location=device)
    m.load_state_dict(chkpt['model'])
    optimizer.load_state_dict(chkpt['optimizer'])
    start_iter = chkpt['iter']
    min_los = chkpt['loss']
    print(f"Շարունակում եմ {start_iter}-րդ քայլից")

else:
    start_iter = 0
    min_los = float('inf')
    if INIT_FROM is not None:
        m.load_state_dict(torch.load(INIT_FROM, map_location=device))
        print(f"Սկսում եմ {INIT_FROM}-ից")



patience = PATIENCE

for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        if losses['val'] <= min_los:
            min_los = losses['val']
            torch.save(m.state_dict(), SAVE_PATH)
            patience = PATIENCE
        else:
            patience -= 1
            if patience == 0:
                break
        torch.save({"model": m.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "iter": iter, "loss": min_los}, LAST_PATH)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {get_lr(iter):.2e}")


    xb, yb = get_batch("train")
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    for g in optimizer.param_groups:
        g['lr'] = get_lr(iter)

    optimizer.step()

m.load_state_dict(torch.load(SAVE_PATH, map_location=device))

Սկսում եմ /content/drive/MyDrive/sevak_bpe200h_best.pth-ից
step 0: train loss 3.8957, val loss 3.7282, lr 0.0001
step 20: train loss 3.7697, val loss 3.6564, lr 9.55e-05
step 40: train loss 3.6825, val loss 3.6115, lr 9.1e-05
step 60: train loss 3.6425, val loss 3.6048, lr 8.65e-05
step 80: train loss 3.5868, val loss 3.6032, lr 8.2e-05
step 100: train loss 3.5478, val loss 3.5729, lr 7.75e-05
step 120: train loss 3.5079, val loss 3.5724, lr 7.3e-05
step 140: train loss 3.4805, val loss 3.5637, lr 6.850000000000001e-05
step 160: train loss 3.4673, val loss 3.5527, lr 6.400000000000001e-05
step 180: train loss 3.4176, val loss 3.5617, lr 5.95e-05
step 200: train loss 3.4173, val loss 3.5572, lr 5.5e-05
step 220: train loss 3.3887, val loss 3.5427, lr 5.05e-05
step 240: train loss 3.3650, val loss 3.5498, lr 4.6e-05
step 260: train loss 3.3651, val loss 3.5466, lr 4.15e-05
step 280: train loss 3.3644, val loss 3.5353, lr 3.7000000000000005e-05
step 300: train loss 3.3223, val loss 3.5432

<All keys matched successfully>

## Generation
Sampling with temperature and top-k

In [ ]:
m = GPT().to(device)
m.load_state_dict(torch.load(f"{DRIVE}/sevak_bpe200j_best.pth", map_location=device))
m.eval()

PROMPT = "Հայաստան"
context = torch.tensor([encode(PROMPT)], device=device)
with torch.no_grad():
    out = m.generate(context, max_new_tokens=200, temperature=1.0, top_k=30)
print(decode(out[0].tolist()))


Հայաստան է կոչվում։

Քար ու քարանձավում
Մոռացած ծիածան է,
Որ այնտեղ էլ դեռ չծնվածն է։

Հարկավոր չէ՞ նաև ինձ համար պարզել,
Որտեղ, որտեղ, ի՜նչ կա,
Որտեղ արդեն... անքնությամբ,
Ուրեմն, ուր որ կա,
Որտեղ, ուր որ է, պիտի,
Մոռացած, սակայն,
Չի՛ լինելու՝ ո՛չ...

Եվ ոչ ես,-
Այլև ոչ այնպես միայն ինձ համար,
Քարից ես չեմ մնալու,
Այլ ինձ համար,
Որքան որ ես չեմ զգալու։
-Խելքըս շատ է դժվար,-
Ես չեմ ուզում բնավ էլ՝
Դատարկաձեռնությամբ պարապ...
Ես քեզ չեմ նայում։

Մի՛ հարց, ինձ մի՛ պատասխանեք,
Ինչպե՞ս ես,
Որ...
